# Module 2: Local LLMs with LM Studio
**หลักสูตร:** การพัฒนาเวิร์กโฟลว์ปัญญาประดิษฐ์สำหรับสร้างข้อความและรูปภาพบนเครื่องคอมพิวเตอร์ส่วนบุคคล  
**(Mastering Local AI Stack: Jupyter Notebook × LM Studio × ComfyUI)**

---

## 🎯 วัตถุประสงค์การเรียนรู้ (Learning Objectives)
1. เข้าใจการเลือกขนาดโมเดล Local LLM (`3B Parameters`, `GGUF Q4_K_M`) และการตั้งค่าพารามิเตอร์ให้เหมาะสมกับเครื่องที่มี **VRAM 6GB (RTX 3050)** หรือ **Apple Silicon Unified Memory 16GB**
2. สามารถเชื่อมต่อกับ **LM Studio Local Server** (`http://localhost:1234/v1`) ด้วยไลบรารีมาตรฐาน `openai` ได้
3. เข้าใจผลกระทบของพารามิเตอร์ `system prompt`, `temperature`, และ `max_tokens`
4. **ปฏิบัติการ (Hands-on Lab):** เขียนโค้ดอ่านบทความจาก `sample_article.txt` แล้วสั่งให้ Local LLM สกัดประเด็นสำคัญออกมาเป็น **JSON** เพื่อนำมาแสดงผลในตาราง `pandas.DataFrame`

---
## ⚙️ รายการตรวจสอบก่อนรันโค้ด (LM Studio Setup)
1. เปิดโปรแกรม **LM Studio** และโหลดโมเดลขนาด 3B เช่น `Qwen2.5-3B-Instruct-GGUF (Q4_K_M)` หรือ `Llama-3.2-3B-Instruct-GGUF (Q4_K_M)` (ใช้ VRAM เพียง ~2.2 GB)
2. ตั้งค่า **Context Length** เป็น `2048` หรือ `4096`
3. ไปที่แถบ **Developer (Local Server)** ด้านซ้าย แล้วกดปุ่ม **Start Server** (พอร์ตเริ่มต้นคือ `1234`)

In [ ]:
from openai import OpenAI

# ตั้งค่า Client เชื่อมต่อไปยัง LM Studio Local Server บนเครื่องของเราเอง
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"  # ใช้ข้อความใดก็ได้เนื่องจากรันแบบ Local ไม่เสียค่าใช้จ่าย
)

# ตรวจสอบรายชื่อโมเดลที่กำลังโหลดอยู่ใน LM Studio
try:
    models = client.models.list()
    loaded_model_id = models.data[0].id if models.data else "local-model"
    print(f"✅ เชื่อมต่อ LM Studio สำเร็จ! โมเดลที่กำลังทำงานคือ: {loaded_model_id}")
except Exception as e:
    loaded_model_id = "local-model"
    print("⚠️ ไม่สามารถเชื่อมต่อ LM Studio ได้ กรุณาตรวจสอบว่ากด Start Server ที่พอร์ต 1234 แล้วหรือยัง")
    print(f"รายละเอียดข้อผิดพลาด: {e}")

## ส่วนที่ 1: การตั้งค่า System Prompt, Temperature และ Max Tokens

- **System Prompt:** คำสั่งกำหนดบทบาท ขอบเขตความรู้ และรูปแบบคำตอบของ AI
- **Temperature (`0.0` - `1.0`):** ค่ายิ่งต่ำ (เช่น `0.1` - `0.3`) คำตอบจะยิ่งแม่นยำ ตรงไปตรงมา เหมาะกับการสรุปข้อมูลหรือสกัด JSON ส่วนค่ายิ่งสูง (เช่น `0.7` - `0.9`) จะเหมาะกับงานคิดสร้างสรรค์
- **Max Tokens:** จำนวน Token สูงสุดที่อนุญาตให้โมเดลสร้างคำตอบออกมา

In [ ]:
response = client.chat.completions.create(
    model=loaded_model_id,
    messages=[
        {
            "role": "system",
            "content": "คุณคืออาจารย์ผู้เชี่ยวชาญด้าน AI ประจำมหาวิทยาลัย อธิบายเรื่องยากให้เข้าใจง่าย กระชับ เป็นภาษาไทย"
        },
        {
            "role": "user",
            "content": "ช่วยอธิบายคำว่า Quantization (เช่น Q4_K_M) ในโมเดล GGUF ให้ฟังภายใน 3 บรรทัดหน่อยครับ"
        }
    ],
    temperature=0.3,
    max_tokens=250
)

print("=== คำตอบจาก Local LLM ===")
print(response.choices[0].message.content)

## ส่วนที่ 2: โจทย์ปฏิบัติการ (Hands-on Lab) — สกัดข้อมูลจากบทความเป็น JSON และแสดงผลด้วย Pandas

ในงานจริง เรามักใช้ Local LLM อ่านเอกสารภายในองค์กรแล้วสกัดข้อมูลออกมาเป็นโครงสร้าง **JSON Array** เพื่อนำไปบันทึกลงฐานข้อมูลหรือสร้างตารางสรุปผล

เราจะอ่านเนื้อหาจากไฟล์ `../assets/sample_article.txt` แล้วสั่งให้โมเดลสกัดประเด็นสำคัญออกมาเป็นรายการ JSON ที่มีฟิลด์:
- `topic`: หัวข้อประเด็นสำคัญ
- `category`: หมวดหมู่ (เช่น Hardware, Model Optimization, Benefits)
- `key_takeaway`: สรุปใจความสำคัญสั้นๆ 1 ประโยค

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd

# 1. อ่านไฟล์บทความภาษาไทย
article_text = Path("../assets/sample_article.txt").read_text(encoding="utf-8")

# 2. กำหนด System Prompt เพื่อบังคับให้ LLM ตอบกลับเป็น JSON เท่านั้น
extraction_system_prompt = """คุณคือระบบสกัดข้อมูลอัจฉริยะ (Information Extraction AI)
หน้าที่ของคุณคืออ่านบทความที่ผู้ใช้ส่งให้ แล้วสกัดประเด็นสำคัญออกมา 4-5 ข้อ
โดยต้องตอบกลับเป็น JSON Array เท่านั้น ห้ามมีคำอธิบายอื่นนำหน้าหรือต่อท้าย
โครงสร้าง JSON ที่ต้องการ:
[
  {
    "topic": "ชื่อหัวข้อ",
    "category": "หมวดหมู่ (Hardware / Model / Workflow / Benefit)",
    "key_takeaway": "สรุปใจความสำคัญภาษาไทย 1 ประโยค"
  }
]"""

extraction_response = client.chat.completions.create(
    model=loaded_model_id,
    messages=[
        {"role": "system", "content": extraction_system_prompt},
        {"role": "user", "content": f"จงสกัดประเด็นสำคัญจากบทความต่อไปนี้:\n\n{article_text}"}
    ],
    temperature=0.1,
    max_tokens=800
)

raw_output = extraction_response.choices[0].message.content.strip()
print("=== Raw Output จาก Local LLM ===")
print(raw_output)

In [ ]:
# 3. แปลงข้อความ JSON ให้กลายเป็นตาราง Pandas DataFrame
def parse_json_safely(text: str) -> list:
    """ฟังก์ชันช่วยสกัดเฉพาะส่วน [...] ในกรณีที่โมเดลเผลอใส่ ```json ติดมาด้วย"""
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\[.*\]", cleaned, flags=re.DOTALL)
    target = match.group(0) if match else cleaned
    return json.loads(target)

try:
    extracted_items = parse_json_safely(raw_output)
    df_summary = pd.DataFrame(extracted_items)
    print("✅ แปลงข้อมูล JSON เป็นตาราง Pandas สำเร็จ!")
    display(df_summary)
except Exception as parse_err:
    print(f"❌ ไม่สามารถแปลง JSON ได้: {parse_err}")

---
### ✅ สรุปท้ายบทเรียน Module 2
ในบทเรียนนี้เราได้เรียนรู้การเชื่อมต่อ **LM Studio Local Server** ด้วยไลบรารี `openai` และการควบคุมให้โมเดลขนาด 3B สกัดข้อมูลจากบทความภาษาไทยออกมาเป็นโครงสร้าง JSON พร้อมแสดงผลในตาราง Pandas

ในช่วงบ่าย (**Module 3** และ **Module 4**) เราจะทำความรู้จักกับ **ComfyUI** และนำความสามารถของ Local LLM มาช่วยแต่ง Prompt สร้างรูปภาพโดยอัตโนมัติ!